# Fine-tuning Whisper on the Wolof Dataset (galsenai/wolof_tts)



In [1]:
# Uncomment and fill in to push to the Hub
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

# HUB_MODEL_ID = "your-username/whisper-small-wolof"
# trainer.push_to_hub(hub_model_id=HUB_MODEL_ID)
# tokenizer.push_to_hub(HUB_MODEL_ID)
# processor.push_to_hub(HUB_MODEL_ID)
# print(f"Model published to : https://huggingface.co/{HUB_MODEL_ID}")

## 1. Install Dependencies



In [2]:
%%capture
# Kaggle Python 3.12: uninstall old version before reinstalling
!pip uninstall -y transformers tokenizers 2>/dev/null
!pip install --upgrade pip

# Consistent versions tested on Python 3.12 + CUDA
!pip install -q transformers==4.44.2
!pip install -q tokenizers==0.19.1
!pip install -q datasets==2.21.0
!pip install -q accelerate==0.34.2
!pip install -q evaluate==0.4.3
!pip install -q jiwer==3.0.4
!pip install -q librosa==0.10.2
!pip install -q soundfile==0.12.1
!pip install -q tensorboard

print('Installation complete')
print('RESTART KERNEL: Runtime > Restart and Clear All Outputs')

## 2. Imports & Configuration

In [3]:
import os
import torch
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import evaluate

# Configuration
MODEL_NAME    = "openai/whisper-small"   # Change to 'medium' if GPU >16GB
TASK          = "transcribe"
SAMPLING_RATE = 16_000
OUTPUT_DIR    = "/kaggle/working/whisper-wolof"
HF_DATASET    = "galsenai/wolof_tts"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

2026-05-12 20:15:25.024961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778616925.254493      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778616925.324608      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778616925.844707      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778616925.844750      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778616925.844753      22 computation_placer.cc:177] computation placer alr

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## 3. Load the Dataset

In [4]:
print("Loading galsenai/wolof_tts ...")
raw_dataset = load_dataset(HF_DATASET, trust_remote_code=True)
print(raw_dataset)
print("\n Exemple :")
print(raw_dataset["train"][0])

Loading galsenai/wolof_tts ...


Generating train split:   0%|          | 0/36009 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4001 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'gender'],
        num_rows: 36009
    })
    test: Dataset({
        features: ['audio', 'text', 'gender'],
        num_rows: 4001
    })
})

 Exemple :
{'audio': {'path': None, 'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
        0.00000000e+00, -3.05175781e-05, -3.05175781e-05]), 'sampling_rate': 16000}, 'text': 'Fan lay sax ?', 'gender': 'male'}


In [5]:
# Resample to 16 kHz (required by Whisper)
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

# If no test split, create one (10%)
if "test" not in raw_dataset:
    split = raw_dataset["train"].train_test_split(test_size=0.1, seed=42)
    raw_dataset = DatasetDict({"train": split["train"], "test": split["test"]})

print(f"Train : {len(raw_dataset['train'])} examples")
print(f"Test  : {len(raw_dataset['test'])} examples")

Train : 36009 examples
Test  : 4001 examples


## 4. Load Whisper Processor

> Note: We load the tokenizer **without `language`**: Wolof (`wo`) is not in Whisper's vocabulary.

In [6]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

# No language="wo" — Wolof is not natively supported
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME)
tokenizer.set_prefix_tokens(language=None, task=TASK)

processor = WhisperProcessor.from_pretrained(MODEL_NAME)

print(f"Processor loaded : {MODEL_NAME}")
print(f"   pad_token_id      : {tokenizer.pad_token_id}")
print(f"   bos_token_id      : {tokenizer.bos_token_id}")
print(f"   eos_token_id      : {tokenizer.eos_token_id}")

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Processor loaded : openai/whisper-small
   pad_token_id      : 50257
   bos_token_id      : 50257
   eos_token_id      : 50257


## 5. Preprocessing Data

In [7]:
def prepare_dataset(batch):
    """
    - Extracts log-mel spectrogram (feature_extractor)
    - Tokenizes text WITHOUT language prefix (add_special_tokens=False)
      then manually wraps with BOS + EOS
    """
    audio = batch["audio"]

    # Feature extraction (80-band log-mel spectrogram)
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Tokenization without language token
    token_ids = tokenizer.encode(batch["text"], add_special_tokens=False)
    batch["labels"] = [tokenizer.bos_token_id] + token_ids + [tokenizer.eos_token_id]
    return batch


print("Preprocessing in progress...")
dataset = raw_dataset.map(
    prepare_dataset,
    remove_columns=raw_dataset["train"].column_names,
    num_proc=2,
    desc="Preprocessing"
)
print("Preprocessing complete")
print(dataset)

Preprocessing in progress...


Preprocessing (num_proc=2):   0%|          | 0/36009 [00:00<?, ? examples/s]

Preprocessing (num_proc=2):   0%|          | 0/4001 [00:00<?, ? examples/s]

Preprocessing complete
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 36009
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 4001
    })
})


## 6. Model + Data Collator

In [8]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad audio features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding tokens with -100 (ignored in loss)
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if present at position 0
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


# Load the model
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Remove ALL language references in config AND generation_config
# Otherwise, model.generate() looks for the <|wo|> token and crashes during evaluation
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

model.generation_config.language = None    # Key: not "wo"
model.generation_config.task = None        # None as well to avoid conflict
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.no_timestamps_token_id = tokenizer.convert_tokens_to_ids("<|notimestamps|>")

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
print("Model + Data Collator ready")
print(f"   decoder_start_token_id : {model.config.decoder_start_token_id}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model + Data Collator ready
   decoder_start_token_id : 50258


## 7. WER Metric

In [9]:
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad_token_id before decoding
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("WER metric configured")

WER metric configured


## 8. Training Configuration

In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Duration
    num_train_epochs=3,

    # Batch & gradient
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # effective batch = 32
    gradient_checkpointing=True,

    # Optimizer
    learning_rate=1e-5,
    warmup_steps=500,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.98,

    # Precision
    fp16=torch.cuda.is_available(),

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Generation
    predict_with_generate=True,
    generation_max_length=225,

    # Logging
    logging_steps=25,
    report_to=["tensorboard"],
    save_total_limit=2,

    # Miscellaneous
    dataloader_num_workers=4,
    push_to_hub=False,
)

print("Training arguments configured")

Training arguments configured


## 9. Start Training

In [11]:
import os
import torch
import numpy as np
from transformers import Seq2SeqTrainer


# 1. Autorisation des objets numpy pour PyTorch 2.6+
try:
    torch.serialization.add_safe_globals([
        np.core.multiarray._reconstruct,
        np.ndarray,
        np.dtype,
        np.core.multiarray.scalar,
    ])
    print("Objets numpy autorisés pour torch.load")
except Exception as e:
    print("Attention :", e)

# 2. Initialisation du Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer._gen_kwargs = {
    "forced_decoder_ids": None,
    "suppress_tokens": [],
}

# 3. Chemins checkpoint

# IMPORTANT :
# Remplace ce chemin par TON vrai chemin Kaggle Input
chemin_source = "/kaggle/input/notebooks/gaussgauss/whisper-wolo-galsenai/whisper-wolof/checkpoint-1126"

# Dossier de travail local
dossier_base = "/kaggle/working/whisper-wolof"
dossier_travail = f"{dossier_base}/checkpoint-1126"

# Création du dossier local
os.makedirs(dossier_base, exist_ok=True)

# 4. Copie du checkpoint vers /working

if os.path.exists(chemin_source):

    print("Copie du checkpoint...")

    os.system(
        f"cp -r '{chemin_source}' '{dossier_base}/'"
    )

    print("Checkpoint copié avec succès.")

else:
    raise FileNotFoundError(
        f"Checkpoint introuvable : {chemin_source}"
    )

# 5. Suppression du fichier RNG problématique
rng_file = os.path.join(dossier_travail, "rng_state.pth")

if os.path.exists(rng_file):
    os.remove(rng_file)
    print("rng_state.pth supprimé")
else:
    print("Pas de rng_state.pth trouvé")

# 6. Vérification des fichiers
print("\nFichiers du checkpoint :")
print(os.listdir(dossier_travail))

# 7. Reprise de l'entraînement
# =========================================================
print("\nReprise de l'entraînement...")

trainer.train(
    resume_from_checkpoint=dossier_travail
)

print("\nEntraînement repris avec succès.")

Objets numpy autorisés pour torch.load


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Copie du checkpoint...
Checkpoint copié avec succès.
rng_state.pth supprimé

Fichiers du checkpoint :
['trainer_state.json', 'config.json', 'optimizer.pt', 'model.safetensors', 'generation_config.json', 'scheduler.pt', 'preprocessor_config.json', 'training_args.bin']

Reprise de l'entraînement...


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to ga

Epoch,Training Loss,Validation Loss,Wer
3,0.468100,0.615388,58.373166


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
Some non-default generation parameters are set in the model config. These should go into a Generati


Entraînement repris avec succès.


## 10. Final Evaluation

In [12]:
print("Evaluating on the test split...")
results = trainer.evaluate()
print(f"Final WER : {results['eval_wer']:.2f}%")
print(results)

Evaluating on the test split...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Final WER : 56.21%
{'eval_loss': 0.637951135635376, 'eval_wer': 56.20765617856104, 'eval_runtime': 2002.1274, 'eval_samples_per_second': 1.998, 'eval_steps_per_second': 0.125, 'epoch': 3.0}


## 11. Save the Model

In [13]:
SAVE_PATH = "/kaggle/working/whisper-wolof-final"
trainer.save_model(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to : {SAVE_PATH}")

# List saved files
for f in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f:40s} {size:.1f} MB")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


Model saved to : /kaggle/working/whisper-wolof-final
  added_tokens.json                        0.0 MB
  config.json                              0.0 MB
  generation_config.json                   0.0 MB
  merges.txt                               0.5 MB
  model.safetensors                        967.0 MB
  normalizer.json                          0.1 MB
  preprocessor_config.json                 0.0 MB
  special_tokens_map.json                  0.0 MB
  tokenizer_config.json                    0.3 MB
  training_args.bin                        0.0 MB
  vocab.json                               1.0 MB


## 12. (Optional) Push to Hugging Face Hub

In [14]:
# Décommentez et remplissez pour pousser sur le Hub
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

# HUB_MODEL_ID = "votre-username/whisper-small-wolof"
# trainer.push_to_hub(hub_model_id=HUB_MODEL_ID)
# tokenizer.push_to_hub(HUB_MODEL_ID)
# processor.push_to_hub(HUB_MODEL_ID)
# print(f"✅ Modèle publié sur : https://huggingface.co/{HUB_MODEL_ID}")

## 13. Quick Inference (Test)

In [15]:
from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model=SAVE_PATH,
    device=0 if DEVICE == "cuda" else -1,
)

# Test on the first 20 examples of the test split
for i in range(20):
    sample       = raw_dataset["test"][i]
    audio_array  = sample["audio"]["array"]
    reference    = sample["text"]
    result       = asr({"raw": audio_array, "sampling_rate": SAMPLING_RATE})
    print(f"[{i}] Reference    : {reference}")
    print(f"[{i}] Transcription: {result['text']}")
    print()

/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


[0] Reference    : Baay Siidi Jóob 773242415
[0] Transcription:  Baay Siidi Jóob

[1] Reference    : Neex na ko.
[1] Transcription:  neex na ko

[2] Reference    : Tom manul a jëfandikoo lii.
[2] Transcription:  Tom manul a jëfandikoo lii

[3] Reference    : Bul jottaliku mukk loo làqul njariñ lu la man a jegeel sa Boroom
[3] Transcription:  Bul jokkli ko mukk lu làqul njariñ, lu la man a jokkaleeg sa boroom

[4] Reference    : Su ngeen dawi jee nag ?
[4] Transcription:  Su ngeen dawi jee nag

[5] Reference    : Waxtaan nal ak Tom mu dimbale la.
[5] Transcription:  Waxtaanal ak Tom mu dimbali la

[6] Reference    : Tangor   ~ tund, jànj.
[6] Transcription:  Tangoor, Tundu, Jànj

[7] Reference    : Niru na lees miin.
[7] Transcription:  nuru na lees miin

[8] Reference    : Duma nelaw mukk ci sama barabu liggéeyukaay.
[8] Transcription:  Duma nelaw mukk ci sama barabu liggéeyukaay



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[9] Reference    : ④- Caytug junniy miliyaar yu «xaalisu mbas mi, céddaleg dund bi
[9] Transcription:  Caytug 14 miliyaar yu xaalisu mbas mi, séddalig dund bi

[10] Reference    : Réew mépp i wor laa
[10] Transcription:  Réew mépp woor la

[11] Reference    : Tubaab yi ngir ñu man koo jàpp lay yii lañu indi woon
[11] Transcription:  Tu baab yi ngir ñu man koo jàpp la yi la ñu àndi woon

[12] Reference    : Ana lumu jëndul ?
[12] Transcription:  Analu mu jëndul

[13] Reference    : Damaa bëgg ngeen woy.
[13] Transcription:  Damaa bëgg ngeen woy.

[14] Reference    : Am na juróomi njiiti bànqaas muy
[14] Transcription:  Am na juróom ñenti bànqaas muy

[15] Reference    : Góor ñépp teew na ñu.
[15] Transcription:  Góoru ñépp teew nañu

[16] Reference    : Daam dañu koo may doomi bàjjanaam ca Kajoor
[16] Transcription:  Daam, dañu koo may doomi bàjjanaam ca kajoor

[17] Reference    : Maa la ëpp.
[17] Transcription:  Ma la ëpp

[18] Reference    : Tànqalug siiwal rekk la woon.
[18] Transcr